# Sanctions Screening: Fuzzy Matching with thefuzz (Spark-Optimized)

This notebook compares **two approaches** for running `thefuzz` on 10,000 holding–sanctioned pairs:

| Approach | How it works | Scalability |
|----------|-------------|-------------|
| **Naive Python** | `df.toPandas()` → sequential loop on driver | Single-node bottleneck; ~30 min reported by customer at scale |
| **Spark `pandas_udf`** | Distributed across workers; batched vectorized execution | Scales linearly with cluster; seconds–minutes for 10K+ |

Scorers used:
- `fuzz.ratio` – basic Levenshtein similarity (0–100)
- `fuzz.token_set_ratio` – best for entity-name matching (handles word reordering, subsets, suffixes)

In [ ]:
%pip install thefuzz -q

In [ ]:
dbutils.library.restartPython()

In [ ]:
import time
from thefuzz import fuzz

CATALOG      = "renjiharold_demo"
SCHEMA       = "sanctions_screening"
SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.holdings_sanctioned_pairs"
RESULT_TABLE = f"{CATALOG}.{SCHEMA}.thefuzz_results"

df = spark.table(SOURCE_TABLE)
n_rows = df.count()
print(f"Loaded {n_rows:,} pairs from {SOURCE_TABLE}")

## Approach 1 — Naive Python (single-node baseline)

Collects everything to the driver with `toPandas()` and runs a sequential loop.  
This replicates what a typical Python-only implementation does and shows the bottleneck.

In [ ]:
pdf = df.toPandas()

start = time.time()
pdf["fuzz_ratio"]           = [fuzz.ratio(h, s)           for h, s in zip(pdf["holding"], pdf["sanctioned"])]
pdf["fuzz_token_set_ratio"] = [fuzz.token_set_ratio(h, s) for h, s in zip(pdf["holding"], pdf["sanctioned"])]
naive_elapsed = time.time() - start

print(f"Naive Python:  {naive_elapsed:.2f}s for {len(pdf):,} rows")
print(f"Projected for 100K rows: ~{naive_elapsed * 10:.0f}s  ({naive_elapsed * 10 / 60:.1f} min)")
print(f"Projected for 1M rows:   ~{naive_elapsed * 100:.0f}s ({naive_elapsed * 100 / 60:.1f} min)")

## Approach 2 — Spark-distributed with `pandas_udf`

Key optimisations over the naive approach:
1. **No `toPandas()`** — data stays distributed across workers.
2. **`pandas_udf`** — processes micro-batches in vectorised fashion on each executor.
3. **`repartition()`** — ensures enough parallelism so all workers are utilised.
4. On **serverless compute**, workers auto-scale to match the workload.

In [ ]:
from pyspark.sql.functions import pandas_udf, col
import pandas as pd

@pandas_udf("int")
def fuzz_ratio_udf(s1: pd.Series, s2: pd.Series) -> pd.Series:
    from thefuzz import fuzz
    return pd.Series(
        [fuzz.ratio(str(a), str(b)) if pd.notna(a) and pd.notna(b) else None
         for a, b in zip(s1, s2)]
    )

@pandas_udf("int")
def fuzz_token_set_ratio_udf(s1: pd.Series, s2: pd.Series) -> pd.Series:
    from thefuzz import fuzz
    return pd.Series(
        [fuzz.token_set_ratio(str(a), str(b)) if pd.notna(a) and pd.notna(b) else None
         for a, b in zip(s1, s2)]
    )

In [ ]:
start = time.time()

result_df = (
    df
    .repartition(8)
    .withColumn("fuzz_ratio",           fuzz_ratio_udf(col("holding"), col("sanctioned")))
    .withColumn("fuzz_token_set_ratio", fuzz_token_set_ratio_udf(col("holding"), col("sanctioned")))
)

result_df.write.mode("overwrite").saveAsTable(RESULT_TABLE)
spark_elapsed = time.time() - start

print(f"Spark pandas_udf: {spark_elapsed:.2f}s for {n_rows:,} rows")
print(f"Speedup vs naive: {naive_elapsed / spark_elapsed:.1f}x")

## Results — high-scoring matches

In [ ]:
display(
    spark.sql(f"""
        SELECT holding, sanctioned, fuzz_ratio, fuzz_token_set_ratio
        FROM {RESULT_TABLE}
        WHERE fuzz_token_set_ratio >= 60
        ORDER BY fuzz_token_set_ratio DESC, fuzz_ratio DESC
    """)
)

In [ ]:
display(
    spark.sql(f"""
        SELECT
            CASE
                WHEN fuzz_token_set_ratio >= 80 THEN '80-100 (High)'
                WHEN fuzz_token_set_ratio >= 60 THEN '60-79  (Medium)'
                WHEN fuzz_token_set_ratio >= 40 THEN '40-59  (Low)'
                ELSE '0-39   (Very Low)'
            END AS score_band,
            count(*) AS pair_count
        FROM {RESULT_TABLE}
        GROUP BY 1
        ORDER BY 1 DESC
    """)
)

## Performance summary

| Factor | Naive Python | Spark `pandas_udf` |
|--------|-------------|---------------------|
| Execution model | Sequential on driver | Distributed across workers |
| Memory | All data on driver (`toPandas()`) | Data stays partitioned |
| Scaling | Linear with row count | Sub-linear — add workers |
| Serverless | N/A | Auto-scales workers |

For the customer's 30-minute problem at 10K rows, the likely root cause is running `process.extract()` (N × M cross-comparisons) in a single Python loop. The Spark approach distributes both the pre-paired comparison **and** can parallelise cross-comparisons by pushing the cartesian join into Spark.